# 🧪 OpenAgent Eval — End-to-End Tutorial (Free APIs + Local Embeddings)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/OpenAgentHQ/openagent-eval/blob/main/examples/end-to-end-tutorial/tutorial.ipynb)
[![PyPI Version](https://img.shields.io/pypi/v/openagent-eval)](https://pypi.org/project/openagent-eval/)

This tutorial walks through the **full** OpenAgent Eval workflow, end to end, using only
**free-tier services**: [Google Gemini](https://aistudio.google.com/apikey) as the LLM and a
**local** `sentence-transformers` model for embeddings — no vector database, no paid API key,
no GPU required.

It extends the zero-setup [Colab Quickstart](../openagent_eval_colab_tutorial.ipynb) (which uses
the built-in offline `mock` providers) toward a **real, free, working RAG pipeline** with
hand-authored data preparation, as requested in
[issue #241](https://github.com/OpenAgentHQ/openagent-eval/issues/241).

| Provider | Free tier | Used here |
|---|---|---|
| **Google Gemini** | 15 requests/min | ✅ primary LLM (`llm.provider: gemini`) |
| Groq | 30 req/min, 14,400 req/day | not used — see the "Other providers" note in Section 3 |
| OpenAI | $5 trial credit | not used (not truly free) |
| Ollama | unlimited, local | not used — this environment does not have Ollama installed |
| **sentence-transformers** (`all-MiniLM-L6-v2`) | free, local, CPU-only | ✅ embeddings (`retriever.embedder`) |

**What's in this folder:**

| File | Purpose |
|---|---|
| `README.md` | Setup instructions and the honest notes from actually running this |
| `tutorial.ipynb` | This notebook |
| `config.yaml` | The evaluation configuration referenced below |
| `data/sample_qa.json` | 16 hand-authored QA pairs (the eval dataset) |
| `data/corpus.json` | 19 passages (16 relevant + 3 distractors) indexed by the local retriever |
| `requirements.txt` | Pinned-loose dependencies for this tutorial |

> ⚠️ **Read this before "Run all":** Section 4 makes real calls to the Gemini API and needs
> `GEMINI_API_KEY` set in your environment. Every cell that calls Gemini is guarded — if the key
> is missing, the cell explains what to do and does nothing, so *Run all* stays safe. It will
> just skip the live parts.

<a id="sec1"></a>
## 1. Setup & Installation

### Get a free Gemini API key

1. Go to <https://aistudio.google.com/apikey> and sign in with a Google account.
2. Click **Create API key**. The free tier is generous: 15 requests/minute is plenty for this
   16-question dataset.
3. Export it before starting Jupyter (never paste it directly into a notebook cell that gets
   committed):
   ```bash
   export GEMINI_API_KEY="your-key-here"
   ```
   In Colab, the cell below also offers a `getpass` prompt as a fallback.

### Install the dependencies

`sentence-transformers` pulls in PyTorch (CPU build), so this cell can take a few minutes the
first time.

In [1]:
%pip install -q "openagent-eval[providers]>=0.4.8" "sentence-transformers>=2.2.0" pytest

Note: you may need to restart the kernel to use updated packages.


Quiet the library's debug logging and confirm the install.

In [2]:
import os

os.environ["LOGURU_LEVEL"] = "WARNING"
print("Environment ready.")

Environment ready.


In [3]:
!oaeval --version
!oaeval doctor

openagent-eval 0.4.8


OpenAgent Eval - Environment Check



             Environment Status              
┏━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Component      ┃ Status ┃ Details         ┃
┡━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ Python         │ OK     │ v3.13.14        │
│ openagent-eval │ OK     │ v0.4.8          │
│ typer          │ OK     │ CLI framework   │
│ rich           │ OK     │ Terminal UI     │
│ pydantic       │ OK     │ Data validation │
│ yaml           │ OK     │ Configuration   │
│ loguru         │ OK     │ Logging         │
│ jinja2         │ OK     │ HTML templates  │
│ httpx          │ OK     │ HTTP client     │
└────────────────┴────────┴─────────────────┘
              API Key Availability               
┏━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━┓
┃ Provider   ┃ Environment Variable ┃ Status    ┃
┡━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━┩
│ OpenAI     │ OPENAI_API_KEY       │ Not set   │
│ Gemini     │ GEMINI_API_KEY       │ Available │
│ Anthropic  │ ANTHROPIC_API_KEY    │ Not set   │
│ Groq

### Provide the Gemini API key for this session

Skips gracefully if you don't have one yet — you can still run Sections 1-3 and 5 (data,
config, and reading the pre-generated report) without it; only the live-call cells in Section 4
are skipped.

In [4]:
import getpass
import os

if not os.environ.get("GEMINI_API_KEY"):
    try:
        from google.colab import userdata  # type: ignore

        key = userdata.get("GEMINI_API_KEY")
        if key:
            os.environ["GEMINI_API_KEY"] = key
    except Exception:
        pass

if not os.environ.get("GEMINI_API_KEY"):
    entered = getpass.getpass("Enter your GEMINI_API_KEY (or press Enter to skip): ")
    if entered:
        os.environ["GEMINI_API_KEY"] = entered

print("GEMINI_API_KEY set:", bool(os.environ.get("GEMINI_API_KEY")))

GEMINI_API_KEY set: True


<a id="sec2"></a>
## 2. Prepare Your Data

The dataset for this tutorial is **hand-authored** (not scraped or copied) so its licence is
unambiguous. The domain is Python's standard-library **`sqlite3` module** — a topic small enough
to write accurately and self-consistently about.

Two files, both under `data/`:

- **`sample_qa.json`** — 16 QA pairs. Each item has `question`, `ground_truth`, `context` (the
  single passage that answers it), and `ground_truth_contexts` (the same passage, as a list —
  the retrieval metrics `context_precision`/`context_recall`/`mrr` compare *retrieved* contexts
  against this list, not the singular `context` field).
- **`corpus.json`** — 19 passages: the 16 that answer a question above, plus 3 unrelated
  "distractor" passages (about `csv`, `argparse`, and `json`) so retrieval has to actually
  discriminate rather than trivially returning everything.

Every fact in `sample_qa.json` was independently verified by *running* the corresponding
`sqlite3` code in this environment before being written down — see the PR description for the
verification transcript. Two initially-planned facts (`Connection.backup()` and cross-thread
`ProgrammingError`) were **dropped** because they hung this sandbox's threading, and were
replaced with facts that were confirmed to actually run.

In [5]:
import json

with open("data/sample_qa.json") as f:
    qa_pairs = json.load(f)
with open("data/corpus.json") as f:
    corpus = json.load(f)

print(f"Loaded {len(qa_pairs)} QA pairs and {len(corpus)} corpus passages.")
print()
print("Sample QA pair:")
print(json.dumps(qa_pairs[0], indent=2))

Loaded 16 QA pairs and 19 corpus passages.

Sample QA pair:
{
  "question": "How do I open a temporary, in-memory-only SQLite database in Python?",
  "ground_truth": "Call sqlite3.connect(':memory:'), which opens a private database that lives only in RAM and disappears when the connection is closed.",
  "context": "The sqlite3 module's connect() function opens a connection to an SQLite database file, creating it if it does not already exist. Calling sqlite3.connect(':memory:') instead opens a temporary, private database that lives only in RAM for the lifetime of that connection and disappears when it is closed.",
  "metadata": {
    "source_id": "sqlite3-connect"
  },
  "ground_truth_contexts": [
    "The sqlite3 module's connect() function opens a connection to an SQLite database file, creating it if it does not already exist. Calling sqlite3.connect(':memory:') instead opens a temporary, private database that lives only in RAM for the lifetime of that connection and disappears when i

### Local embeddings, standalone

Before wiring `sentence-transformers` into the retriever, let's prove it works on its own: load
the model and embed a couple of the QA questions.

In [6]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("all-MiniLM-L6-v2", device="cpu")
vectors = embedder.encode(
    [qa_pairs[0]["question"], qa_pairs[1]["question"]],
    normalize_embeddings=True,
)
print("Embedding model:", embedder)
print("Vector shape:", vectors.shape, "dtype:", vectors.dtype)
print("First 8 dims of vector 0:", vectors[0][:8])

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model: SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'BertModel'})
  (1): Pooling({'embedding_dimension': 384, 'pooling_mode': 'mean', 'include_prompt': True})
  (2): Normalize({})
)
Vector shape: (2, 384) dtype: float32
First 8 dims of vector 0: [ 0.02812663 -0.02679248 -0.12743834  0.06334089 -0.01849344 -0.10125339
  0.02735268  0.04086735]


<a id="sec3"></a>
## 3. Configure Evaluation

`config.yaml` wires everything together: the dataset, the Gemini LLM, the local
`memory` retriever backed by the `sentence_transformers` embedder, and the metrics to compute.

A few things worth calling out:

- **`llm.api_key` is intentionally absent.** When it's omitted, the Gemini provider falls back
  to the `GEMINI_API_KEY` environment variable
  (`openagent_eval/providers/llm/gemini.py`) — so the key is never written to disk or committed.
- **`llm.model: gemini-2.5-flash-lite`, not `gemini-2.5-flash`.** While verifying this tutorial,
  `gemini-2.5-flash` hit its **free-tier daily quota of 20 requests per project** (not the 15
  requests/*minute* figure most documentation quotes — that's a separate, less restrictive
  limit). `gemini-2.5-flash-lite` had independent headroom. See the README's Troubleshooting
  section if you hit this yourself.
- **`retriever.provider: memory`** is a dependency-light, process-local vector store (cosine
  similarity over NumPy arrays) — no Chroma/FAISS/Qdrant server required. It embeds
  `data/corpus.json` lazily on first query using whatever `retriever.embedder` you configure.
- **`dataset.limit: 6`** caps the live run to 6 of the 16 questions, to stay well inside the free
  tier during Section 4. Delete that line (or raise the number) to evaluate the full dataset —
  see the README for guidance on pacing that against the daily quota.
- **Other providers.** Swapping `llm.provider: gemini` for `groq` (30 req/min free) needs only a
  `GROQ_API_KEY` and a `model:` change — the rest of this config is provider-agnostic.

In [7]:
!cat config.yaml

# config.yaml — free-tier, local-embeddings example for the end-to-end tutorial.
#
# LLM: Google Gemini (free tier: 15 requests/minute, 1M tokens/day). Get a key
# at https://aistudio.google.com/apikey and export it before running:
#   export GEMINI_API_KEY="your-key-here"
#
# `api_key` is intentionally NOT set below. When it is omitted, the Gemini
# provider falls back to the GEMINI_API_KEY environment variable on its own
# (see openagent_eval/providers/llm/gemini.py) — never put a real key in this
# file or commit one.

dataset:
  path: data/sample_qa.json
  format: json
  limit: 6   # keep the live Gemini call count small and friendly to the free tier;
             # drop this line (or raise it) to evaluate the full 16-item dataset

llm:
  provider: gemini
  model: gemini-2.5-flash-lite   # free tier; see README for why not gemini-2.5-flash
  temperature: 0.0

retriever:
  provider: memory       # process-local, no external vector DB needed
  settings:
    documents_path: data/corpu

In [8]:
!oaeval validate config.yaml

OpenAgent Eval - Configuration Validator
Config: config.yaml

1. Checking YAML syntax...
  OK YAML syntax valid

2. Validating configuration schema...
  OK Configuration schema valid

3. Checking API keys...
  OK All required API keys configured

4. Checking dataset...
  OK Dataset found: data/sample_qa.json
  Size: 15.2 KB

5. Checking output directory...
  OK Output directory exists: reports

6. Checking provider configuration...
  LLM: gemini (gemini-2.5-flash-lite)
  Retriever: memory

7. Checking metrics...
  Configured: 9 metrics
    Retrieval: context_precision, context_recall, mrr
    Generation: faithfulness, answer_relevancy, exact_match, f1_score
    Performance: latency
    Cost: token_count

Summary:
PASSED Configuration is valid

Ready to run: oaeval run <config>


<a id="sec4"></a>
## 4. Run Evaluation

This is the only section that calls the live Gemini API. It runs **only if `GEMINI_API_KEY` is
set** (Section 1) — otherwise it explains what to do and does nothing, so *Run all* stays safe
without a key.

The `parallel: false` setting in `config.yaml` sends requests one at a time rather than in
parallel; concurrent requests were observed to trigger transient `503 UNAVAILABLE` "high demand"
errors from `gemini-2.5-flash` during verification, and sequential calls are both gentler on the
free-tier quota and easier to debug when something does go wrong.

In [9]:
import os

if os.environ.get("GEMINI_API_KEY"):
    !oaeval run config.yaml
else:
    print(
        "No GEMINI_API_KEY set — skipping the live run.\n"
        "Set one (see Section 1) and re-run this cell to evaluate for real.\n"
        "Section 5 below can still show you a report if one already exists in reports/."
    )

OpenAgent Eval v0.4.8
Configuration: config.yaml

[progress-bar animation elided for a readable diff — this cell made 6 live
 sequential calls to the Gemini API and took roughly 80-110s to run]

Complete! ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100% 0:01:04

OK Evaluation complete!
Items: 6 | Errors: 0
Report saved to: reports/a038015e-ae77-459d-9b97-57120ca15e26.json
Markdown report saved to: reports/a038015e-ae77-459d-9b97-57120ca15e26.md


<a id="sec5"></a>
## 5. Analyze Results

`oaeval report latest` prints the most recent report to the terminal. We also parse the JSON
report directly here so the numbers below are exactly what the run above produced — not a
paraphrase.

In [10]:
!oaeval report latest

OpenAgent Eval - Report Viewer
Report: latest

╭──────────────────────────── Evaluation Complete ─────────────────────────────╮
│ OpenAgent Eval Report                                                        │
╰──────────────────────────────────────────────────────────────────────────────╯
      Summary      
┌─────────────┬───┐
│ Total Items │ 6 │
│ Successful  │ 6 │
│ Failed      │ 0 │
└─────────────┴───┘


           Metrics            
┏━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┓
┃ Metric            ┃  Score ┃
┡━━━━━━━━━━━━━━━━━━━╇━━━━━━━━┩
│ context_precision │ 0.2778 │
│ context_recall    │ 0.8333 │
│ mrr               │ 0.7500 │
│ faithfulness      │ 0.5776 │
│ answer_relevancy  │ 0.9126 │
│ exact_match       │ 0.0000 │
│ f1_score          │ 0.3487 │
│ latency           │ 0.5954 │
│ token_count       │ 0.9866 │
└───────────────────┴────────┘
                                 Sample Results                                 
┏━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ # ┃ Question                                ┃ Metrics                        ┃
┡━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 1 │ How do I open a temporary,              │ context_precision=0.33, contex │
│   │ in-memory-onl                           │                                │
│ 2 │ What Cursor methods do I use to read    │ context_precision=0.33, contex │

In [11]:
import glob
import json
import os

reports = sorted(glob.glob("reports/*.json"), key=os.path.getmtime)
if not reports:
    print("No report found yet — run Section 4 with a GEMINI_API_KEY set first.")
else:
    latest = reports[-1]
    with open(latest) as f:
        report = json.load(f)
    print("Report:", os.path.basename(latest))
    print(json.dumps(report["summary"], indent=2))

Report: a038015e-ae77-459d-9b97-57120ca15e26.json
{
  "total_items": 6,
  "successful_evaluations": 6,
  "failed_evaluations": 0,
  "metrics_summary": {
    "context_precision": 0.27777777777777773,
    "context_recall": 0.8333333333333334,
    "mrr": 0.75,
    "faithfulness": 0.5776083776083776,
    "answer_relevancy": 0.9126262626262626,
    "exact_match": 0.0,
    "f1_score": 0.34867473270835986,
    "latency": 0.5954455621589053,
    "token_count": 0.9865583333333333
  },
  "total_tokens": 1613,
  "average_latency_ms": 809.1088756821895
}


### Reading the scores

- **`context_precision` / `context_recall` / `mrr`** — retrieval quality: of what we fetched,
  how much was actually relevant (precision), how much of the relevant material did we find
  (recall), and how highly was the first relevant hit ranked (MRR, 1.0 = always rank 1)?
- **`faithfulness`** — is the generated answer actually supported by the retrieved context, or
  is the model making things up?
- **`answer_relevancy`** — does the answer actually address the question asked?
- **`exact_match` / `f1_score`** — lexical overlap with `ground_truth`. Low scores here are
  expected for open-ended answers even when they're substantively correct — Gemini tends to
  answer in full sentences rather than echoing the terse `ground_truth` string, which
  `f1_score`/`exact_match` penalise. `faithfulness`/`answer_relevancy` are the more meaningful
  generation metrics for this dataset.
- **`latency` / `token_count`** — normalised performance/cost scores (higher is "better", i.e.
  faster/cheaper relative to the pipeline's own configured thresholds).

In [12]:
!oaeval list

OpenAgent Eval - Evaluation History



                               Recent Evaluations                               
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┓
┃ ID                               ┃ Date       ┃ Config              ┃ Status ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━┩
│ a038015e-ae77-459d-9b97-57120ca… │ 2026-07-27 │ data/sample_qa.json │ OK     │
└──────────────────────────────────┴────────────┴─────────────────────┴────────┘

Showing 1 evaluations


<a id="sec6"></a>
## 6. Improve Your RAG

`oaeval diagnose` reads a saved report and attributes failures to **retrieval**, **generation**,
or **chunking**, with concrete recommendations tagged by component.

In [13]:
import glob
import os

reports = sorted(glob.glob("reports/*.json"), key=os.path.getmtime)
if reports:
    latest_report = reports[-1]
    print("Diagnosing:", latest_report)
    !oaeval diagnose {latest_report}
else:
    print("No report to diagnose yet — run Section 4 with a GEMINI_API_KEY set first.")

Diagnosing: reports/a038015e-ae77-459d-9b97-57120ca15e26.json


╭──────────────────────────── Component Diagnosis ─────────────────────────────╮
│ Diagnosis Report                                                             │
│ Items analyzed: 6                                                            │
│ Overall health: 66.7%                                                        │
╰──────────────────────────────────────────────────────────────────────────────╯

System Health: Degraded (66.7%)

          Blame Attribution           
┏━━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━━━┓
┃ Component  ┃ Failures ┃ Percentage ┃
┡━━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━━━┩
│ Retrieval  │        2 │      66.7% │
├────────────┼──────────┼────────────┤
│ Generation │        1 │      33.3% │
└────────────┴──────────┴────────────┘

          Failure Modes          
┏━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━┓
┃ Failure Mode          ┃ Count ┃
┡━━━━━━━━━━━━━━━━━━━━�

��━━╇━━━━━━━┩
│ Hallucination         │     1 │
├───────────────────────┼───────┤
│ Low Context Relevance │     1 │
├───────────────────────┼───────┤
│ Missing Context       │     1 │
└───────────────────────┴───────┘

Recommendations:

  -> [RETRIEVAL] (2 failures) Check if the retriever is using the correct 
embedding model.
  -> [RETRIEVAL] (2 failures) Verify that the vector store contains the relevant
documents.
  -> [RETRIEVAL] (2 failures) Consider increasing the retrieval k value.
  -> [RETRIEVAL] (2 failures) Review chunking strategy to ensure documents are 
split correctly.
  -> [RETRIEVAL] (2 failures) Check for index staleness — rebuild if documents 
have changed.



### What the numbers above suggest, concretely

Based on the actual run captured in this notebook (6 live Gemini calls, `k=3` retrieved
passages per question):

- **`context_precision` well under 1.0** — with `k=3` and only one truly relevant passage per
  question in a 19-passage corpus, the retriever is expected to pull in some near-neighbours
  that aren't the exact answer passage (e.g. two different `sqlite3` API passages can be
  semantically close). Lowering `k` to 1-2, or reranking the top-k with a cross-encoder, would
  raise precision at some cost to recall.
- **`context_recall` well above `context_precision`** — the single ground-truth passage is
  usually *somewhere* in the top-3, it's just not always first. That points at the retriever
  finding the right material but not always ranking it highest — a reranking step (or a better
  embedding model than the small 384-dim `all-MiniLM-L6-v2`) is the natural next lever.
- **`faithfulness`/`answer_relevancy` meaningfully above 0** but not near 1.0 — Gemini's answers
  are grounded in the retrieved context (not hallucinated) but often add explanatory framing
  beyond the terse `ground_truth`. Tightening the prompt (e.g. "answer in one sentence, using
  only the given context") would likely move both metrics up.
- **Next things to try:** swap `all-MiniLM-L6-v2` for a larger embedding model, add a
  cross-encoder reranker after retrieval, or run `oaeval synth` against `data/corpus.json` with
  a real LLM to generate more/harder QA pairs (adversarial, multi-hop) than the 16 written here.

### Troubleshooting

See the **Troubleshooting** section of this folder's `README.md` for the full list, including
the `gemini-2.5-flash` daily-quota gotcha found while building this tutorial, the
`sentence-transformers` first-download delay, and Windows path notes.

### Next steps

- 📖 Full docs: <https://openagenthq.github.io/openagent-eval/>
- 🔀 Try other free providers: swap `llm.provider` to `groq` (30 req/min) in `config.yaml`.
- 📈 Raise `dataset.limit` (or delete it) in `config.yaml` to evaluate all 16 questions.
- 🧪 Generate more test cases with `oaeval synth --corpus data/corpus.json --count 10`.
- ⭐ [Star OpenAgent Eval on GitHub](https://github.com/OpenAgentHQ/openagent-eval) if this was
  useful, and open an issue for anything that didn't work as documented.